In [10]:
import numpy as np
import math


In [11]:
def euclidean_distance(x1: np.array, x2: np.array):
    diff = x1 - x2
    sq = diff**2
    return np.sqrt(sq.sum(axis=-1))


def pairwise_euclidean_distance(X: np.array):
    A = X[:, None, :]
    B = X[None, :, :]
    diff = A-B
    sq = diff**2
    return np.sqrt(sq.sum(axis=-1))


def pairwise_manhattan_distance(X: np.array):
    A = X[:, None, :]
    B = X[None, :, :]
    diff = np.abs(A-B)
    return diff.sum(axis=-1)


def pairwise_cosine_distance(X: np.array):
    # Normalize each row to unit length
    X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
    
    # Cosine similarity matrix
    cos_sim = X_norm @ X_norm.T
    
    return 1 - cos_sim


In [27]:
class KNNClassifier:
    def __init__(self, k: int = 3, dist="euclidean"):
        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be a positive integer, got {k!r}.")
        self.k = k
        self.dist = dist
        self.X_train = None
        self.y_train = None
    
    def pairwise_euclidean_distance(self, X: np.array):
        A = self.X_train[None, :, :]    # (1, n_train, d)
        B = X[:, None, :]               # (n_test, 1, d)
        diff = A-B
        sq = diff ** 2
        return np.sqrt(sq.sum(axis=-1)) # (n_test, n_train)
    
    def pairwise_manhattan_distance(self, X: np.array):
        A = self.X_train[None, :, :]    # (1, n_train, d)
        B = X[:, None, :]               # (n_test, 1, d)
        diff = np.abs(A-B)
        return diff.sum(axis=-1)        # (n_test, n_train)
    
    def pairwise_cosine_distance(self, X: np.array):
        X_train = self.X_train

        # Add tiny epsilon to avoid division by zero
        eps = 1e-12
        X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)              # (n_test, d)
        X_train_norm = X_train / (np.linalg.norm(X_train, axis=1, keepdims=True) + eps)  # (n_train, d)

        # Cosine similarity: (n_test, d) @ (d, n_train) -> (n_test, n_train)
        cos_sim = X_norm @ X_train_norm.T

        # Convert similarity to distance
        return 1 - cos_sim
    
    def compute_distance(self, X: np.array):
        if self.dist == "euclidean":
            return self.pairwise_euclidean_distance(X)
        elif self.dist == "manhattan":
            return self.pairwise_manhattan_distance(X)
        elif self.dist == "cosine":
            return self.pairwise_cosine_distance(X)
        else:
            raise Exception(f"Error: {self.dist} is not a valid distance.")

    def _check_fitted(self):
        if self.X_train is None or self.y_train is None or self.classes is None:
            raise RuntimeError("KNNClassifier is not fitted yet. Call 'fit(X, y)' first.")
        
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features), got shape {X.shape}.")
        if y.ndim != 1:
            raise ValueError(f"y must be 1D (n_samples,), got shape {y.shape}.")
        if X.shape[0] == 0:
            raise ValueError("X is empty.")
        if X.shape[0] != y.shape[0]:
            raise ValueError(
                f"X and y must have the same number of samples, got {X.shape[0]} and {y.shape[0]}."
            )
        if self.k > X.shape[0]:
            raise ValueError(
                f"k={self.k} is greater than the number of training samples n={X.shape[0]}."
            )
        
        self.X_train = X
        self.classes, self.y_train = np.unique(y, return_inverse=True)
        return self
    
    def predict(self, X):
        self._check_fitted()
        X = np.asarray(X)
        if X.ndim != 2:
            raise ValueError(f"X must be 2D (n_samples, n_features), got shape {X.shape}.")

        dists = self.compute_distance(X)    # shape (n_test, n_train)
        if dists.shape != (X.shape[0], self.X_train.shape[0]):
            raise RuntimeError(
                f"Distance matrix shape mismatch: expected {(X.shape[0], self.X_train.shape[0])}, "
                f"got {dists.shape}."
            )
        
        preds = []
        for i in range(dists.shape[0]):
            nn_idx = np.argsort(dists[i])[:self.k]
            nn_labels = self.y_train[nn_idx]
            
            # voting
            counts = np.bincount(nn_labels, minlength=len(self.classes))
            pred = np.argmax(counts)
            preds.append(pred)
        
        preds_int = np.array(preds, dtype=int)
        return self.classes[preds_int]
    


### Testing the Algorithm


In [35]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [36]:
print("\n" + "="*50)
print("TEST 1: Benchmark vs Scikit-Learn (Breast Cancer)")
print("="*50)

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Custom Model...")
my_model = KNNClassifier(k=5, dist="euclidean")
my_model.fit(X_train, y_train)

print("Training Sklearn Model...")
sk_model = KNeighborsClassifier(n_neighbors=5, weights="uniform", metric="minkowski", p=2) # minkowski with p=2 is basically the euclidean distance
sk_model.fit(X_train, y_train)

my_pred = my_model.predict(X_test)
sk_pred = sk_model.predict(X_test)

my_acc = accuracy_score(y_test, my_pred)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"\nRESULTS:")
print(f"Custom KNN Accuracy: {my_acc:.4f}")
print(f"Sklearn KNN Accuracy: {sk_acc:.4f}")

# Verification logic
if abs(my_acc - sk_acc) < 0.05:
    print(">> SUCCESS: Performance is comparable to production library.")
else:
    print(">> WARNING: Significant performance deviation.")



TEST 1: Benchmark vs Scikit-Learn (Breast Cancer)
Training Custom Model...
Training Sklearn Model...

RESULTS:
Custom KNN Accuracy: 0.9561
Sklearn KNN Accuracy: 0.9561
>> SUCCESS: Performance is comparable to production library.
